In [2]:
import pandas as pd
from PIL import Image
import io
import torch
from typing import List, Optional, Any
from langchain_core.embeddings import Embeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document
from qdrant_client import QdrantClient
from qdrant_client.http import models
from models.scold.model import LVL
from transformers import RobertaTokenizer
from torchvision import transforms

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LVL()
model.load_state_dict(torch.load("models/scold/scold.pt", map_location=device))
model.to(device)
model.eval()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Andakara\AppData\Local\Temp\ipykernel_184628\2083541103.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe

LVL(
  (image_encoder): ImageEncoder(
    (swin): FeatureListNet(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
        (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      )
      (layers_0): SwinTransformerStage(
        (downsample): Identity()
        (blocks): Sequential(
          (0): SwinTransformerBlock(
            (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=128, out_features=384, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=128, out_features=128, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(dim=-1)
            )
            (drop_path1): Identity()
            (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=128, out_fe

In [4]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [5]:
class SCOLDEmbeddings(Embeddings):
    """Custom SCOLD embeddings for text and image data."""
    
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
    
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """Generate embeddings for a list of text documents."""
        inputs = self.tokenizer(
            texts, 
            return_tensors="pt", 
            padding=True, 
            truncation=True
        ).to(self.device)
        
        with torch.no_grad():
            text_emb = self.model.get_texts_feature(
                inputs["input_ids"], 
                inputs["attention_mask"]
            )
        
        return text_emb.cpu().numpy().tolist()
    
    def embed_query(self, text: str) -> List[float]:
        """Generate embedding for a single query text."""
        return self.embed_documents([text])[0]

In [6]:
class ImageEmbeddings(Embeddings):
    """Custom image embeddings using SCOLD model."""
    
    def __init__(self, model, transform, device):
        self.model = model
        self.transform = transform
        self.device = device
    
    def embed_documents(self, image_bytes_list: List[bytes]) -> List[List[float]]:
        """Generate embeddings for a list of image bytes."""
        embeddings = []
        
        for image_bytes in image_bytes_list:
            image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            image_tensor = self.transform(image).unsqueeze(0).to(self.device)
            
            with torch.no_grad():
                image_emb = self.model.get_images_features(image_tensor)
            
            embeddings.append(image_emb.cpu().numpy()[0].tolist())
        
        return embeddings
    
    def embed_query(self, image_bytes: bytes) -> List[float]:
        """Generate embedding for a single image query."""
        return self.embed_documents([image_bytes])[0]

In [7]:
df = pd.read_parquet("data/leafnet/leafnet_sampled.parquet")

In [8]:
text_embeddings = SCOLDEmbeddings(model, tokenizer, device)
image_embeddings = ImageEmbeddings(model, transform, device)

In [9]:
client = QdrantClient(url="http://localhost:6333")
collection_name = "leaf_disease_collection_langchain"

In [10]:
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(size=512, distance=models.Distance.COSINE),
    )

In [ ]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=text_embeddings, 
)


In [12]:
documents = []
for idx, row in df.iterrows():
    text_doc = Document(
        page_content=str(row['caption']),
        metadata={
            "type": "text",
            "source_id": idx
        }
    )
    documents.append(text_doc)
    
    image_doc = Document(
        page_content=row['image']['bytes'],  
        metadata={
            "type": "image",
            "caption": str(row['caption']),
            "source_id": idx
        }
    )
    documents.append(image_doc)


ValidationError: 1 validation error for Document
page_content
  Input should be a valid string, unable to parse raw data as a unicode string [type=string_unicode, input_value=b'\x89PNG\r\n\x1a\n\x00\x...0\x00\x00IEND\xaeB`\x82', input_type=bytes]
    For further information visit https://errors.pydantic.dev/2.11/v/string_unicode